# NEXA Foundation Model - Training Environment

This notebook is designed to train the NEXA Foundation Model on Google Colab using GPU/TPU resources. It connects to Google Drive for persistent dataset loading and checkpoint saving.

In [ ]:
# 1. Setup Environment
!git clone https://github.com/nexa-ai/nexa-fm.git || echo 'Repository already cloned'
%cd nexa-fm
!pip install torch datasketch PyPDF2 langdetect

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE_DIR = '/content/drive/MyDrive/NEXA_FM'
import os
os.makedirs(f"{DRIVE_BASE_DIR}/datasets/shards", exist_ok=True)
os.makedirs(f"{DRIVE_BASE_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_BASE_DIR}/logs", exist_ok=True)
print("Google Drive configured at:", DRIVE_BASE_DIR)

In [ ]:
# 3. Initialize Training Engine
import torch
from backend.models.nexa_fm.training_engine.config import TrainingConfig
from backend.models.nexa_fm.training_engine.trainer import Trainer
# from backend.models.nexa_fm.training_engine.dataloader import ShardDataLoader
from backend.models.nexa_fm.architecture import NexaFMModel
from backend.models.nexa_fm.config import NexaFMConfig
from backend.models.tokenizer.bpe_tokenizer import BPETokenizer

# Initialize Configurations
model_config = NexaFMConfig.tiny() # Use Tiny config for validation/testing
train_config = TrainingConfig(
    checkpoint_dir=f"{DRIVE_BASE_DIR}/checkpoints",
    log_dir=f"{DRIVE_BASE_DIR}/logs",
    batch_size=8,
    gradient_accumulation_steps=4
)

# Load Model
model = NexaFMModel(model_config)

# NOTE: You must provide a valid dataloader instance below once datasets are available.
# dataloader = ShardDataLoader(...)
dataloader = [] # Placeholder

# Initialize Trainer
trainer = Trainer(model, train_config, dataloader)
# trainer.resume_from_checkpoint()

In [ ]:
# 3.5 Dry-Run Validation
# Run this cell to execute a single forward and backward pass to validate the pipeline.
try:
    success = trainer.dry_run()
    if success:
        print("\n🟢 READY FOR TRAINING")
    else:
        print("\n🔴 FAILED")
except Exception as e:
    print("\n🔴 FAILED:", e)

In [ ]:
# 4. Start Training
try:
    trainer.train()
except KeyboardInterrupt:
    print("Training interrupted. Saving final checkpoint...")
    trainer.checkpoint_manager.save(trainer.model, trainer.optimizer, trainer.scheduler, trainer.global_step, trainer.epoch, trainer.config)
    print("Checkpoint saved.")